**1. What challenges did you encounter when provisioning and executing machine learning workflows in a managed cloud environment (such as Colab Enterprise or Compute Engine) compared to local or standard free-tier notebooks?**

At first, the project/organization configuration of Colab Enterprise was unclear to me, and I didn't realize I had to create my own project. Once this was resolved, I didn't have many other difficulties. It was however confusing to determine how to manage runtime and stop my session so that I didn't accidentally burn through credits, since there is no clear 'Stop' button outside of the Manage Session tab within the Runtime tab. It was very nested, I thought.

**2. How did you identify and handle implicit missing values (such as -200 sentinel indicators) across the sensor columns, and how did your imputation strategy affect model stability?**

When I first checked the dataset, it looked complete, but the summary statistics showed that every numeric column had a minimum of exactly -200, rather than a null value. The dataset uses -200 as a placeholder for missing values, so I counted these in each column, replaced them with proper blank values, and filled the blanks with each column's median. I chose the median because a few very high readings would have pulled the average upward. This brought the data back to realistic ranges and let the models train smoothly, with consistent results. The main drawback is that columns with a lot of missing data, especially NMHC and the CO target, ended up with many identical filled-in values, which makes what the model learned from them less reliable and therefore less stable.

**3. Explain how the Scikit-Learn ColumnTransformer and Pipeline streamline the data preparation process and prevent data leakage across preprocessing and training steps in your cloud pipeline.**

The Scikit-Learn ColumnTransformer and Pipeline streamline the data prepration process by simplifying what would otherwise be a strenous and time-consuming mathametical undertaking into a few lines of code. Instead of having to manually scale values and convert categorical data to numeric values to scale them as well, we can use Scikit-Learn to make the transformation happen very quickly. These tools also help prevent data leakage by consolidating the transformation/pipeline actions to the training data and applying them consistently to the test data.

**4. How did comparing StandardScaler versus MinMaxScaler impact your model training convergence, gradient descent optimization, and final evaluation metrics?**

StandardScaler worked better than MinMaxScaler for both models. With MinMaxScaler, both models needed more iterations to finish training, took slightly longer, and ended up with lower accuracy and less reliable probability predictions. SGDClassifier was affected the most. The likely reason is that some features have a few extreme readings, and MinMaxScaler squeezes the rest of those values into a very small range near zero. Gradient descent then makes only tiny progress on those features, so it needs more passes over the data to reach a good result. StandardScaler centers each feature around zero and gives them all a similar spread, so gradient descent moves toward the best solution more directly. Overall, StandardScaler is the better choice for this dataset, especially for models trained with gradient descent.

**5. How does configuring SGDClassifier differ from standard closed-form solvers, and what advantages do probability outputs and evaluation metrics provide when assessing classification performance?**

With SGDClassifier, I had to make several training choices myself: how big each learning step is, how many passes to make over the data, and when to stop. Choosing these well affects both how fast the model trains and how good the result is. A standard solver like LogisticRegression handles those decisions automatically and works through the whole dataset at once, so it needs very little tuning. SGD’s advantage is that it learns a little at a time from each example, which makes it practical for very large datasets. In my results, the two models had similar accuracy, but SGD finished in fewer passes. Probability outputs were useful because they show how confident the model is, not just which class it picked. Log-loss uses those probabilities, and it revealed that the softmax model’s predictions were more trustworthy, even though accuracy alone made the two models look almost the same. Looking at several measures together gives a fuller picture of how well a model is really performing.

**6. Based on your regularized regression models (Ridge versus Lasso), how did adjusting the regularization hyperparameter affect feature selection and coefficient shrinkage, and what operational environmental insights do these findings reveal?**

Raising the penalty strength changed the two models in different ways. With Ridge, small values had almost no effect. At larger values, all the coefficients gradually shrank toward zero, but none disappeared entirely, so every feature stayed in the model. Lasso dropped features completely as the penalty grew. The time-of-day, day-of-week, and month columns went first, then most of the sensor and weather features, until only benzene was left, and finally nothing at all. When two features carried nearly the same information, Lasso tended to keep one and drop the other. From an environmental point of view, the features that lasted longest were benzene and nitrogen oxides, which suggests CO rises and falls together with these pollutants, most likely because they share a common source such as traffic exhaust. The time columns were dropped early, so the daily pattern of pollution seems to be already captured by the pollutant readings themselves. Weather played only a small role. In practice, this suggests CO levels could be estimated reasonably well from a smaller set of related measurements, which could make air quality monitoring simpler and cheaper.